<a href="https://colab.research.google.com/github/eeiselem/Circulant-Graphs-n--2-k-labeling/blob/main/CirculantGraphKLabeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problem 3

## research paper mathematical approach

In [ ]:
import math

def cr_labeling(n):
    V = [0] * n
    V[0:4] = [1, 2, 3, 4]
    r=n-3
    diff = n - r
    dfact = math.floor(diff / 2)

    t = 4
    m = 4

    while t < n:
        V[t] = m + 1
        E = edge_weights(V, t + 1, n, dfact)

        if not weight_duplicate(E, t + 1, n, dfact):
            t += 1
            m += 1
        else:
            m += 1

    return V,E, max(V)

def edge_weights(V, t, n, dfact):
    E = [['' for _ in range(t)] for _ in range(t)]

    for i in range(t - dfact - 1):
        for j in range(dfact + i + 1, t):
            if i <= dfact - 1 and j >= n - dfact + i:
                E[i][j] = ''
            else:
                E[i][j] = V[i] + V[j]

    return E

def weight_duplicate(E, t, n, dfact):
    for i in range(1, t - dfact - 1):
        for j in range(dfact + i + 1, t - 1):
            for l in range(i):
                for w in range(j + 1, t):
                    if E[i][j] != '' and E[i][j] == E[l][w]:
                        return True
    return False
for n in range(5,11):
  V,E, K = cr_labeling(n)
  print(f"  Optimal k found : {K}")
  print(V)

  Optimal k found : 5
[1, 2, 3, 4, 5]
  Optimal k found : 8
[1, 2, 3, 4, 6, 8]
  Optimal k found : 12
[1, 2, 3, 4, 6, 9, 12]
  Optimal k found : 18
[1, 2, 3, 4, 6, 9, 13, 18]
  Optimal k found : 27
[1, 2, 3, 4, 6, 9, 13, 19, 27]
  Optimal k found : 36
[1, 2, 3, 4, 6, 9, 13, 19, 28, 36]


## Algorithm 3B

In [ ]:
import itertools
import math
import copy
import time

# Global Variables
BEST_K = float('inf')
BEST_V_LABELS = None
BEST_LABEL_SUM = float('inf')
NODE_COUNT = 0

def get_edges_for_step(current_node_index, n):
    # produces empty array of proper size for n
    edges = []
    if current_node_index >= 2:
        for i in range(current_node_index - 1):
            if not (i == 0 and current_node_index == n - 1):
                edges.append((i, current_node_index))
    return edges


def solve_cr_bnb(current_node_index, current_max_label, n, current_v_labels, used_labels, current_edge_sums):
    # Recursively traverses implicit tree to minimize k, uses sum of labels as tie-breaker for best set of labels
    global BEST_K, BEST_V_LABELS, BEST_LABEL_SUM, NODE_COUNT
    NODE_COUNT += 1

    # two base cases
    # if current max label of set of labels is worse than best k backtrack
    if current_max_label >= BEST_K:
      return
    # if labeled all nodes
    if current_node_index == n:
        current_solution_k = current_max_label
        # if k for current set of labels is better than existing k
        if current_solution_k < BEST_K:
            BEST_K = current_solution_k
            BEST_V_LABELS = copy.deepcopy(current_v_labels)
            # calculates sum for future tie braking if needed in future. that's why te deepcopy is necessary
            BEST_LABEL_SUM = sum(label for label in current_v_labels)
        # if k for current labels is the same as existing k
        elif current_solution_k == BEST_K:
            current_sum = sum(label for label in current_v_labels)
            # compare to stored sum of best solution and replace if current is less than stored
            if current_sum < BEST_LABEL_SUM:
                BEST_V_LABELS = copy.deepcopy(current_v_labels)
                BEST_LABEL_SUM = current_sum
        return

    # find smallest label available, will use later for choosing which nodes are necessary to explore in next level
    start_label = 1
    while start_label in used_labels:
      start_label += 1

    for label_val in itertools.count(start=start_label):
        potential_k = max(current_max_label, label_val)
        # with label_val the k is a worse option than k of best option found so far
        if potential_k > BEST_K:
          break
        # if label_val already in labels continue on to next label_val
        if label_val in used_labels:
          continue

        # returns empty array structure
        new_edges = get_edges_for_step(current_node_index, n)
        sums_ok = True
        sums_added_this_step = set() # Store new sums temporarily for backtracking
        try:
            # loop through edges connecting to previous nodes
            for i, node_j in new_edges:
                v_i = current_v_labels[i]
                # calculate potential edge sum
                current_sum = v_i + label_val
                # if edge repeated sums_ok is false and will trty new label_val
                if current_sum in current_edge_sums:
                  sums_ok = False
                  break
                # if unique so far add to temporary set of sums
                sums_added_this_step.add(current_sum)
        except Exception as e:
          # handles errors if try doesn't work
          sums_ok = False

        # if labels holding to constrain of no repeats
        if sums_ok:
            # assign valid label to current 'node'/'vertices' in label array
            current_v_labels[current_node_index] = label_val
            used_labels.add(label_val) # add labels to used labels for this path
            current_edge_sums.update(sums_added_this_step) # add unique edges to set for this path
            # recursively call this function to traverse down a layer
            solve_cr_bnb(current_node_index + 1, potential_k, n, current_v_labels, used_labels, current_edge_sums)

            # backtracking, reverses changes made above
            used_labels.discard(label_val)
            current_edge_sums.difference_update(sums_added_this_step)
            current_v_labels[current_node_index] = None


if __name__ == "__main__":
    for n in range(5, 8):
        print(f"\nFor n = {n} ---")

        # Initialize globals
        # _,_,BEST_K = cr_labeling(n) #this is using the optimal k based on proffesor asims paper. does not work for n=5 and r=2
        BEST_K = n * (n - 3)/2 * math.log2(n) # based on the upper bound found in proffessor Asims paper
        BEST_V_LABELS = None
        BEST_LABEL_SUM = float('inf')
        NODE_COUNT = 0

        # Initialize local state
        initial_v_labels = [None] * n
        initial_v_labels[0] = 1
        initial_used_labels = {1}
        initial_edge_sums = set()
        initial_max_k_so_far = 1
        start_node = 1

        print(f"  Initial BEST_K set to: {BEST_K}")

        start_time = time.perf_counter()
        solve_cr_bnb(start_node, initial_max_k_so_far, n, initial_v_labels, initial_used_labels, initial_edge_sums)
        end_time = time.perf_counter()
        execution_time = end_time - start_time

        # Print results
        print(f"  Nodes Visited : {NODE_COUNT:,}")
        print(f"  Execution Time: {execution_time:.4f} seconds")
        if BEST_V_LABELS:
            print(f"  Optimal k found : {BEST_K}")
            print(f"  Label Sum for Optimal k: {BEST_LABEL_SUM}")
            print("\n  Edge Sum Matrix: ")
            print(f"  Optimal Vertex Labels: {BEST_V_LABELS}")
            placeholder = "-"

            # Create matrix data structure for printing
            matrix_data = [[placeholder for _ in range(n)] for _ in range(n)]

            # Fill matrix
            # these two for loops replicate adjacency matrix for Cn,r=n-3, along with exclusion of upper right corner
            for j in range(2, n):
                for i in range(j - 1):
                    if not (i == 0 and j == n - 1):
                        edge_sum = BEST_V_LABELS[i] + BEST_V_LABELS[j]
                        matrix_data[i][j] = edge_sum

            # Find width of largest values in matrix
            val = 0
            for r in matrix_data:
                 for c in r:
                      if isinstance(c, (int, float)): # Check if it's a number
                           val = max(c,val)
            width = len(str(val)) + 1

            # Print matrix
            for i in range(n):
                line_parts = []
                for j in range(n):
                    if j >= i : # Position is in upper triangle or diagonal
                        item_to_print = matrix_data[i][j]
                        line_parts.append(f"{str(item_to_print):>{width}}")
                    else: # Position is in lower triangle
                        line_parts.append(f"{'':>{width}}") # Add padding spaces
                print("  " + "".join(line_parts))

        else:
            print("  No valid labeling found.")


For n = 5 ---
  Initial BEST_K set to: 11.60964047443681
  Nodes Visited : 26
  Execution Time: 0.0001 seconds
  Optimal k found : 5
  Label Sum for Optimal k: 15

  Edge Sum Matrix: 
  Optimal Vertex Labels: [1, 2, 3, 4, 5]
   - - 4 5 -
     - - 6 7
       - - 8
         - -
           -

For n = 6 ---
  Initial BEST_K set to: 23.264662506490403
  Nodes Visited : 108
  Execution Time: 0.0004 seconds
  Optimal k found : 6
  Label Sum for Optimal k: 21

  Edge Sum Matrix: 
  Optimal Vertex Labels: [1, 4, 2, 5, 3, 6]
    -  -  3  6  4  -
       -  -  9  7 10
          -  -  5  8
             -  - 11
                -  -
                   -

For n = 7 ---
  Initial BEST_K set to: 39.302968908806456
  Nodes Visited : 1,537
  Execution Time: 0.0087 seconds
  Optimal k found : 9
  Label Sum for Optimal k: 34

  Edge Sum Matrix: 
  Optimal Vertex Labels: [1, 4, 7, 2, 8, 3, 9]
    -  -  8  3  9  4  -
       -  -  6 12  7 13
          -  - 15 10 16
             -  -  5 11
                -  -